# **SpaceX  Falcon 9 First Stage Landing Prediction**

## Interactive Visual Analytics with Folium


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and I discovered some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis notebooks, I visualized that the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this notebook, I performed more interactive visual analytics using `Folium`.


## Objectives


*   Marking all launch sites on a map
*   Marking the success/failed launches for each site on the map
*   Calculating the distances between a launch site to its proximities

This steps allowed me to find some geographical patterns about launch sites.


First, importing required Python packages:


In [5]:
!pip install -q folium


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: C:\Users\p1a2r\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [6]:
import folium
import pandas as pd

In [7]:
# Importing folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Importing folium MousePosition plugin
from folium.plugins import MousePosition
# Importing folium DivIcon plugin
from folium.features import DivIcon

## Marking all launch sites on a map

First, I added each site's location on a map using site's latitude and longitude coordinates


In [9]:
# Downloading and reading the `spacex_launch_geo.csv`
spacex_df=pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv')

Now, taking a look at what are the coordinates for each site.


In [10]:
# Selecting relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


As seen, coordinates are just plain numbers that can not give any intuitive insights about where are those launch sites. So, I visualized those locations by pinning them on a map.


Creating a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [48]:
# Starting location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

I used `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate.


In [49]:
# Creating a red circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color="#ea1f08", fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Creating a red circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Creating an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#ea1f08;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

Now, adding a circle for each launch site in data frame `launch_sites`


Creating and adding `folium.Circle` and `folium.Marker` for each launch site on the site map


In [51]:
# Initial the map
site_map = folium.Map(location=nasa_coordinate, zoom_start=4.5)
# For each launch site, adding a Circle object based on its coordinate (Lat, Long) values. In addition, add Launch site name as a popup label
for launch_site, site_lat, site_long in zip(launch_sites_df['Launch Site'], launch_sites_df['Lat'], launch_sites_df['Long']):
    site_coordinate = [site_lat, site_long]
    
    circle = folium.Circle(site_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup(launch_site))
    
    marker = folium.map.Marker(
        site_coordinate,
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % launch_site,
            )
        )
    site_map.add_child(circle)
    site_map.add_child(marker)
site_map


Now, exploring the map by zoom-in/out the marked areas, and trying to answer the following questions:

*   Are all launch sites in proximity to the Equator line?
*   Are all launch sites in very close proximity to the coast?


## Analysis

- Most of the Launch sites considered in this project are in proximity to the Equator line. Launch sites are made at the closest point possible to Equator line, because anything on the surface of the Earth at the equator is already moving at the maximum speed of 1670 kilometers per hour. 

- All launch sites are in very close proximity to the coast. While starting rockets towards the ocean, we minimise the risk of exploding and failure near populated cities.

# Marking the success/failed launches for each site on the map

Next, I enhanced the map by adding the launch outcomes for each site, and see which sites have high success rates.

In [26]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


Next, I created markers for all launch records.
If a launch was successful `(class=1)`, then I used a green marker and if a launch was failed, I used a red marker `(class=0)`to classify and color code it.


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters are a good way to simplify a map containing many markers having the same coordinate.


First, creating a `MarkerCluster` object


In [52]:
marker_cluster = MarkerCluster()


Then, created a new column in `spacex_df` dataframe called `marker_color` to store the marker colors based on the `class` value


In [53]:
# Applying a function to check the value of `class` column
# If class=1, marker_color value will be green
# If class=0, marker_color value will be red
marker_list= []
for i, Class in enumerate(spacex_df["class"]):
    if int(Class) == 0:
        marker_list.append("red")
    elif int(Class) == 1:
        marker_list.append("green")
spacex_df["marker_color"]= marker_list
spacex_df.head(5)

,Launch Site,Lat,Long,class,marker_color
0,CCAFS LC-40,28.562302,-80.577356,0,red
1,CCAFS LC-40,28.562302,-80.577356,0,red
2,CCAFS LC-40,28.562302,-80.577356,0,red
3,CCAFS LC-40,28.562302,-80.577356,0,red
4,CCAFS LC-40,28.562302,-80.577356,0,red


For each launch result in `spacex_df` data frame, I added a `folium.Marker` to `marker_cluster`

In [54]:
# Adding the Marker cluster to the site map
site_map.add_child(marker_cluster)

# for each row in spacex_df data frame
# creating a Marker object with its coordinate
# and customize the Marker's icon property to indicate if this launch was successed or failed
for site_lat, site_long, marker_color in zip(spacex_df['Lat'], spacex_df['Long'], spacex_df['marker_color']):
    site_coordinate = [site_lat, site_long]
    marker = folium.map.Marker(
        site_coordinate,
        # Creating an icon as a text label
        icon=folium.Icon(color='white', 
                         icon_color=marker_color)
    )
    marker.add_to(marker_cluster)

site_map

From the color-labeled markers in marker clusters, it makes it easy to identify which launch sites have relatively high success rates.


# Calculating the distances between a launch site to its proximities

Next, I explored and analyzed the proximities of launch sites.


First, I added a `MousePosition` on the map to get coordinate for a mouse over a point on the map.


In [55]:
# Adding Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

Now, I zoomed in to a launch site and explore its proximity to find an railway, highway, and coastline.

In [56]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    #radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

I marked down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [57]:
# coordinate of the closet coastline
coastline_coord= [28.56871, -80.60739]
launch_coor= [28.57382, -80.64671]
distance_coastline = calculate_distance(launch_coor[0], launch_coor[1], coastline_coord[0], coastline_coord[1])

In [59]:
# Creating and adding a folium.Marker on the selected closest coastline point on the map
# Displaying the distance between coastline point and launch site using the icon property 

marker = folium.map.Marker(
        coastline_coord,
        # Creating an icon as a text label
        icon=DivIcon(
            icon_size=(400, 400),
            icon_anchor=(0, 0),
            html='<div style="font-size:400; color:#0c10f2;"><b>%s</b></div>' % str(round(distance_coastline, 2))+' km',
            )
    )
marker.add_to(site_map)

site_map

Drawing a `PolyLine` between a launch site to the selected coastline point


In [60]:
# Create a `folium.PolyLine` object using the coastline coordinates and launch site coordinate
folium.PolyLine([coastline_coord, launch_coor], color='blue').add_to(site_map)
site_map

Similarly, I drew a line betwee a launch site to its closest city, railway, highway, etc. I used `MousePosition` to find the their coordinates on the map first


In [61]:
# Creating a marker with distance to a closest city, railway, highway, etc.
# Drawing a line between the marker to the launch site
# Creating a marker with distance to a closest city, coastline, highway, etc.
# Drawing a line between the marker to the launch site
city   = [28.61261, -80.80797]
railway = [28.55752, -80.80155]
highway   = [28.54134, -80.85154]

city_distance = calculate_distance(city[0], city[1], launch_coor[0], launch_coor[1])
railway_distance = calculate_distance(railway[0], railway[1], launch_coor[0], launch_coor[1])
highway_distance = calculate_distance(highway[0], highway[1], launch_coor[0], launch_coor[1])

colors = ['red','orange','green']
html_colors = ['#dc3545','#fd7e14','#198754']

for coordinate ,distance, color, html_color in zip([city, railway, highway], [city_distance, railway_distance, highway_distance], colors, html_colors):
    marker = folium.map.Marker(
            coordinate,
            # Create an icon as a text label
            icon=DivIcon(
                icon_size=(20,20),
                icon_anchor=(0,0),
                html='<div style="font-size: 12; color:'+html_color+';"><b>%s</b></div>' % str(round(distance, 2)) + 'km',
                )
            )
    marker.add_to(site_map)
    folium.PolyLine([coordinate, launch_coor], color=color).add_to(site_map)
site_map

## Analysis
<ul>
<li>From the visual analysis of the launch site KSC LC-39A, I found that it is:</li>
<ul><li>relative close to railway (15.23 km)</li>
    <li>relative close to highway (20.33 km)</li>
    <li>relative close to coastline (3.88 km)</li>
</ul>
<li>Also the launch site KSC LC-39A is relative close to its closest city Titusville (16.33 km).</li>
<li> Failed rocket with its high speed can cover distances like 15-20 km in few seconds. It could be potentially dangerous to Titusville populated areas. </li>
</ul>

## Author

- [Parshv Patel](https://www.linkedin.com/in/parshv-patel-65a90326b) 


<!--## Change Log--!>


<!--| Date (YYYY-MM-DD) | Version | Changed By      | Change Description      |
| ----------------- | ------- | -------------   | ----------------------- |
| 2022-11-09        | 1.0     | Pratiksha Verma | Converted initial version to Jupyterlite|--!>
